# Trabalho Prático — Algoritmos de Ordenação
### Estrutura de Dados · 4º período

---

| | |
|---|---|
| **Aluno(s):** | Arthur Silva Teixeira, Rafael Campello Soares |
| **Matrícula(s):** | 2024021870, 2024017457 |
| **Data de entrega:** | 29/10/2026 |

---

## 1. Introdução

Ordenar uma coleção de elementos segundo uma relação de ordem é um dos problemas mais estudados da Ciência da Computação. Além de ser útil por si só (relatórios, classificações, *rankings*), a ordenação é etapa de preparação para outros algoritmos: a busca binária exige dados ordenados, a eliminação de duplicatas e a junção de tabelas em bancos de dados se tornam mais simples e eficientes, e muitos algoritmos geométricos e de grafos começam ordenando a entrada (CORMEN et al., 2012; ZIVIANI, 2007).

Apesar de o problema ser simples de enunciar, não existe um "melhor" algoritmo de ordenação em sentido absoluto. Os algoritmos clássicos diferem no **custo assintótico** (quadrático, $n\log n$ ou linear), no **uso de memória** (*in-place* ou com vetor auxiliar), na **estabilidade** (preservar a ordem relativa de chaves iguais) e nas **hipóteses sobre a entrada** (apenas comparações, ou inteiros em intervalo limitado, ou números reais uniformemente distribuídos). Além disso, a notação assintótica esconde constantes que, em tamanhos moderados de entrada, podem inverter a ordem de desempenho prevista pela teoria. Por isso, estudar ordenação exige confrontar **análise teórica** e **medições experimentais**.

**Objetivo.** Este trabalho implementa nove algoritmos de ordenação (Bubble Sort, Selection Sort, Insertion Sort, Shellsort, Merge Sort, Quicksort, Counting Sort, Radix Sort e Bucket Sort), instrumentados para contar comparações e movimentações de elementos, e compara seu comportamento teórico e experimental em diferentes tamanhos e tipos de entrada. Também são estudadas as variantes de três algoritmos: as sequências de *gaps* do Shellsort (Shell, Knuth e Ciura), as estratégias de pivô do Quicksort (último elemento, aleatório e mediana de três) e a base do Radix Sort.

**Perguntas que o trabalho responde.**

1. Os tempos e as contagens de operações medidos seguem as classes de complexidade previstas pela análise (quadrática, $n\log n$, linear)? Os expoentes estimados nos gráficos log-log concordam com os esperados?
2. Para entradas pequenas, algoritmos quadráticos podem ser mais rápidos que os $O(n\log n)$? A partir de que tamanho $n_0$ o cruzamento ocorre, e como as constantes ocultas o explicam?
3. Qual o impacto da escolha do pivô no Quicksort, em especial para entradas ordenadas, inversas e com muitos valores repetidos?
4. Como a sequência de *gaps* altera o desempenho do Shellsort?
5. Em que condições os algoritmos lineares (Counting, Radix e Bucket) compensam, e o que acontece quando suas hipóteses sobre a entrada são violadas?
6. Contagem de operações e tempo de execução são métricas equivalentes? Qual delas é mais adequada para validar a teoria e qual para escolher um algoritmo em um sistema real?

**Organização do relatório.** A Seção 2 apresenta a fundamentação teórica: descrição dos algoritmos, recorrências, aplicação do Teorema Mestre, o limite inferior para ordenação por comparação e a tabela consolidada de complexidades. A Seção 3 descreve a metodologia (ambiente, decisões de implementação, geração das entradas, experimentos e medição de tempo). A Seção 4 apresenta os resultados; a Seção 5, a análise crítica; e a Seção 6, as conclusões. Por fim, listam-se as referências.

## 2. Fundamentação teórica

### 2.1 Conceitos e notação

Seja $v$ um vetor com $n$ elementos. Dizemos que o par de índices $(i, j)$, com $i < j$, é uma **inversão** se $v[i] > v[j]$. Um vetor ordenado tem $0$ inversões; um vetor inverso tem $\binom{n}{2} = \frac{n(n-1)}{2}$; um vetor aleatório tem, em média, $\frac{n(n-1)}{4}$. O número de inversões $I$ é uma medida do "grau de desordem" e aparece diretamente no custo de vários algoritmos.

Um algoritmo é **estável** se elementos com chaves iguais mantêm, na saída, a mesma ordem relativa que tinham na entrada. É ***in-place*** se usa apenas $O(1)$ de memória extra (admitindo-se, por convenção, a pilha de recursão em $O(\log n)$). É **comparativo** se obtém a ordem exclusivamente comparando pares de elementos.

Em todo o trabalho, "comparações" são comparações entre **elementos** da entrada (não entre índices) e "trocas" são movimentações de elementos: cada *swap* conta uma unidade e cada atribuição que escreve um elemento em uma posição também conta uma unidade.

### 2.2 Algoritmos quadráticos

Os três algoritmos desta família mantêm uma região ordenada que cresce uma posição por iteração do laço externo. Como cada iteração pode custar $\Theta(n)$, o custo típico é $\Theta(n^2)$.

**Bubble Sort.** Percorre o vetor comparando pares adjacentes e trocando-os quando estão fora de ordem; ao fim da passada $i$, os $i$ maiores elementos estão em suas posições finais. Com a **parada antecipada** (encerrar se uma passada não troca nada), o melhor caso é $\Theta(n)$. Cada troca de vizinhos elimina exatamente uma inversão, logo o número de trocas é igual a $I$. No pior caso (vetor inverso) faz $\frac{n(n-1)}{2}$ comparações e a mesma quantidade de trocas. É estável e *in-place*.

**Selection Sort.** Na iteração $i$, seleciona o menor elemento de $v[i..n-1]$ e o coloca na posição $i$ com uma única troca. Faz sempre $\sum_{i=0}^{n-2}(n-1-i) = \frac{n(n-1)}{2}$ comparações, independentemente da entrada, e no máximo $n-1$ trocas. É *in-place*, mas **não estável**: a troca a longa distância pode inverter a ordem de elementos iguais.

**Insertion Sort.** Pega o próximo elemento (a chave) e o insere na posição correta da região já ordenada, deslocando os maiores para a direita. O número de deslocamentos é exatamente $I$, e o custo é $\Theta(n + I)$: linear para vetores ordenados ou quase ordenados e quadrático para vetores inversos ou aleatórios. É estável e *in-place*. Essa propriedade é explorada pelo Shellsort e pelo Bucket Sort.

### 2.3 Algoritmos subquadráticos

**Teorema Mestre.** Para $T(n) = a\,T(n/b) + f(n)$, com $a \ge 1$ e $b > 1$, compara-se $f(n)$ com $n^{\log_b a}$: (1) se $f(n) = O(n^{\log_b a - \varepsilon})$ para algum $\varepsilon > 0$, então $T(n) = \Theta(n^{\log_b a})$; (2) se $f(n) = \Theta(n^{\log_b a})$, então $T(n) = \Theta(n^{\log_b a}\log n)$; (3) se $f(n) = \Omega(n^{\log_b a + \varepsilon})$ para algum $\varepsilon > 0$ e $a f(n/b) \le c f(n)$ para algum $c < 1$, então $T(n) = \Theta(f(n))$ (CORMEN et al., 2012). O teorema **não** se aplica a recorrências que não têm essa forma, como $T(n) = T(n-1) + \Theta(n)$ ou $T(n) = T(n/10) + T(9n/10) + \Theta(n)$; nesses casos usam-se expansão ou árvore de recursão.

**Shellsort.** Generaliza o Insertion Sort: ordena por inserção os elementos separados por uma distância $h$ (*gap*), para uma sequência decrescente de *gaps* terminando em $h = 1$ (SHELL, 1959). Gaps grandes movem elementos por longas distâncias; quando se chega a $h = 1$, o vetor já está quase ordenado e a passada final é barata. Foram usadas três sequências:

| Sequência | Definição | Pior caso conhecido |
|---|---|---|
| Shell (1959) | $\lfloor n/2\rfloor, \lfloor n/4\rfloor, \dots, 1$ | $\Theta(n^2)$ |
| Knuth (1973) | $h_1 = 1$, $h_{k+1} = 3h_k + 1$, enquanto $h < n/3$ | $O(n^{3/2})$ |
| Ciura (2001) | $1, 4, 10, 23, 57, 132, 301, 701, 1750$, estendida por $\lfloor 2{,}25\,h\rfloor$ | desconhecido |

A sequência de Shell é ruim quando $n$ é potência de 2, porque todos os *gaps* maiores que 1 são pares e as posições pares e ímpares não interagem até a passada final (KNUTH, 1998). O limite $O(n^{3/2})$ para a sequência de Knuth é devido a Knuth e Pratt (KNUTH, 1998; PRATT, 1979). A sequência de Ciura foi obtida empiricamente e é considerada a melhor na prática (CIURA, 2001), mas não há limite de pior caso demonstrado. A análise do caso médio é problema em aberto para a maioria das sequências. O Shellsort é *in-place* e **não é estável**.

**Merge Sort.** Divide o vetor ao meio, ordena recursivamente cada metade e as **intercala** em tempo linear, usando um vetor auxiliar. A recorrência é $T(n) = 2T(n/2) + \Theta(n)$. Com $a = 2$, $b = 2$ e $f(n) = \Theta(n)$, temos $n^{\log_b a} = n = \Theta(f(n))$, e o **caso 2** do Teorema Mestre dá $T(n) = \Theta(n\log n)$. Como a divisão não depende dos valores e a intercalação percorre sempre os $n$ elementos, melhor, médio e pior caso coincidem. Usa $\Theta(n)$ de memória extra e é estável, desde que, em caso de empate, o elemento da metade esquerda seja o primeiro a ser copiado.

**Quicksort.** Escolhe um **pivô**, particiona o vetor (menores à esquerda, maiores à direita) e ordena recursivamente as duas partes (HOARE, 1962). Usamos a partição de Lomuto: o pivô é colocado na última posição e um laço move para a esquerda os elementos $\le$ pivô. As recorrências são:

- *Melhor caso* (partições equilibradas): $T(n) = 2T(n/2) + \Theta(n) = \Theta(n\log n)$, pelo caso 2 do Teorema Mestre.
- *Pior caso* (uma parte vazia): $T(n) = T(n-1) + \Theta(n)$. O Teorema Mestre não se aplica, pois o subproblema diminui por uma constante e não por um fator. Por expansão, $T(n) = c\sum_{k=1}^{n}k + T(0) = \Theta(n^2)$. Com pivô no último elemento, isso ocorre em vetores ordenados, inversos e com todos os elementos iguais.
- *Partição fixa $1:9$:* $T(n) = T(n/10) + T(9n/10) + \Theta(n)$. Pela árvore de recursão, cada nível custa no máximo $cn$ e há entre $\log_{10}n$ e $\log_{10/9}n$ níveis, de modo que $T(n) = \Theta(n\log n)$. Qualquer proporção constante de divisão preserva $n\log n$.
- *Caso médio*: com pivô uniformemente aleatório, o número esperado de comparações é $2(n+1)H_n - 4n \approx 2n\ln n \approx 1{,}39\,n\log_2 n$, em que $H_n = \sum_{k=1}^{n}1/k$ (CORMEN et al., 2012). A prova usa o fato de que dois elementos $z_i < z_j$ são comparados se, e somente se, o primeiro pivô escolhido entre $z_i, \dots, z_j$ for um deles, com probabilidade $2/(j-i+1)$.

Foram implementadas três estratégias de pivô (último, aleatório e mediana de três). A recursão é feita apenas na menor das duas partes, tratando-se a maior com um laço, o que limita a profundidade da pilha a $\lfloor\log_2 n\rfloor$ mesmo no pior caso (tempo $\Theta(n^2)$ inalterado). O Quicksort é *in-place* (usa a pilha) e não é estável.

### 2.4 Limite inferior e algoritmos lineares

**Limite inferior para ordenação por comparação.** Todo algoritmo que ordena apenas por comparações pode ser descrito por uma árvore de decisão binária cujas folhas são as permutações de saída. Para ordenar corretamente toda entrada, a árvore precisa de pelo menos $n!$ folhas; como uma árvore binária de altura $h$ tem no máximo $2^h$ folhas, $h \ge \log_2 n!$. Como $n! \ge (n/2)^{n/2}$,
$$h \;\ge\; \frac{n}{2}\log_2\frac{n}{2} = \Omega(n\log n).$$
Portanto nenhum algoritmo puramente comparativo tem pior caso melhor que $\Omega(n\log n)$ (CORMEN et al., 2012). Os algoritmos a seguir escapam desse limite porque **não comparam elementos entre si**: usam os valores como índices ou para calcular posições. Em troca, exigem hipóteses sobre a entrada.

**Counting Sort.** Hipótese: chaves inteiras em $[0, k)$. Conta as ocorrências de cada valor, transforma as contagens em somas prefixas e percorre a entrada de trás para frente, colocando cada elemento em sua posição final; esse percurso reverso torna o algoritmo estável. Custo $\Theta(n + k)$ de tempo e de espaço, linear quando $k = O(n)$. Se $k$ é muito maior que $n$ (por exemplo $k = n^2$), o termo $k$ domina e a memória pode se tornar proibitiva.

**Radix Sort (LSD).** Hipótese: inteiros não negativos com no máximo $d$ dígitos na base $b$. Ordena do dígito menos significativo para o mais significativo, usando a cada passada o Counting Sort com $k = b$, que precisa ser **estável** para preservar a ordem obtida nas passadas anteriores. Custo $\Theta(d(n + b))$, com $d = \lfloor\log_b M\rfloor + 1$ para o maior valor $M$. Bases maiores reduzem o número de passadas, mas aumentam o vetor de contagem de cada passada. Na implementação deste trabalho, os valores são deslocados pelo mínimo $m$ antes da extração dos dígitos (o dígito de $x$ na passada $i$ é $\lfloor (x - m)/b^i\rfloor \bmod b$). Com isso, números negativos também são aceitos e o número de passadas passa a depender da amplitude: $d = \lfloor\log_b (M - m)\rfloor + 1$, e nenhuma passada é feita se todos os valores forem iguais.

**Bucket Sort.** Hipótese: reais em $[0,1)$ distribuídos uniformemente. Distribui os $n$ elementos em $n$ baldes (o elemento $x$ vai para o balde $\lfloor nx\rfloor$), ordena cada balde com Insertion Sort e concatena. Com $n_i$ o tamanho do balde $i$, o custo é $\Theta(n) + \sum_i O(n_i^2)$. Sob distribuição uniforme, $n_i \sim \text{Binomial}(n, 1/n)$ e $E[n_i^2] = \text{Var}[n_i] + E[n_i]^2 = (1 - 1/n) + 1 = 2 - 1/n$, de modo que o caso médio é $\Theta(n)$. Se todos os elementos caírem no mesmo balde, o custo é $\Theta(n^2)$.

### 2.5 Tabela consolidada de complexidades

| Algoritmo | Melhor | Médio | Pior | Espaço extra | Estável | *In-place* | Recorrência / observação |
|---|---|---|---|---|---|---|---|
| Bubble Sort | $\Theta(n)$ | $\Theta(n^2)$ | $\Theta(n^2)$ | $O(1)$ | Sim | Sim | trocas $= I$ (inversões) |
| Insertion Sort | $\Theta(n)$ | $\Theta(n^2)$ | $\Theta(n^2)$ | $O(1)$ | Sim | Sim | custo $\Theta(n + I)$ |
| Selection Sort | $\Theta(n^2)$ | $\Theta(n^2)$ | $\Theta(n^2)$ | $O(1)$ | Não | Sim | $\frac{n(n-1)}{2}$ comparações sempre; $\le n-1$ trocas |
| Shellsort (Ciura) | $\Theta(n\log n)$ | em aberto | desconhecido | $O(1)$ | Não | Sim | passadas de inserção com *gaps* decrescentes |
| Merge Sort | $\Theta(n\log n)$ | $\Theta(n\log n)$ | $\Theta(n\log n)$ | $\Theta(n)$ | Sim | Não | $2T(n/2) + \Theta(n)$; Mestre, caso 2 |
| Quicksort (aleatório) | $\Theta(n\log n)$ | $\Theta(n\log n)$ (esperado) | $\Theta(n^2)$ | $O(\log n)$ (pilha) | Não | Sim | pior: $T(n-1) + \Theta(n)$ (Mestre não se aplica) |
| Counting Sort | $\Theta(n+k)$ | $\Theta(n+k)$ | $\Theta(n+k)$ | $\Theta(n+k)$ | Sim | Não | inteiros em $[0,k)$ |
| Bucket Sort | $\Theta(n)$ | $\Theta(n)$ (uniforme) | $\Theta(n^2)$ | $\Theta(n)$ | Sim | Não | reais em $[0,1)$ uniformes |
| Radix Sort | $\Theta(d(n+b))$ | $\Theta(d(n+b))$ | $\Theta(d(n+b))$ | $\Theta(n+b)$ | Sim | Não | $d = \lfloor\log_b (M - m)\rfloor + 1$ ($M$ e $m$: maior e menor valor) |

## 3. Metodologia

### 3.1 Ambiente

Todos os experimentos foram executados em uma única máquina, descrita abaixo. As informações foram obtidas com a célula de código seguinte.

| Item | Valor |
|---|---|
| Sistema operacional | Debian GNU/Linux 13 (trixie), kernel 6.12.113+deb13-amd64 |
| Processador | Intel Xeon E5-2630 v4 @ 2,20 GHz (Broadwell-EP, turbo até 3,1 GHz; caches L1 640 KiB, L2 2,5 MiB, L3 25 MiB) |
| Núcleos lógicos | 20 (10 físicos com Hyper-Threading) |
| Memória RAM | 16 GiB DDR4 2400 MHz (um único módulo, canal único) |
| Python | 3.13.5 |
| numpy / pandas / matplotlib | 2.5.4 / 3.1.0 / 3.11.2 |

In [ ]:
# 🧰 Informações do ambiente (copie a saída para a tabela acima)
import platform, os
import numpy as np, pandas as pd, matplotlib
print("Sistema:", platform.system(), platform.release())
print("Processador:", platform.processor() or platform.machine())
print("Núcleos lógicos:", os.cpu_count())
print("Python:", platform.python_version())
print("numpy", np.__version__, "| pandas", pd.__version__, "| matplotlib", matplotlib.__version__)
# RAM: informe manualmente (ou use psutil, se instalado)

### 3.2 Decisões de implementação

Todos os algoritmos foram implementados do zero, sem qualquer rotina de ordenação pronta (`list.sort`, `sorted`, `heapq`, `bisect`, `numpy.sort`). A função `sorted()` é usada **apenas** nos testes de corretude, para gerar o gabarito. Todos recebem um vetor `v` e um objeto contador `c`, e devolvem o vetor ordenado em ordem não decrescente.

**Instrumentação.** Um objeto `Contador` guarda `comp` (comparações) e `trocas` (movimentações). A contagem é feita por funções auxiliares: `menor` e `menor_ou_igual` incrementam `comp` a cada comparação entre elementos; `trocar` incrementa `trocas` em 1 por *swap*; `escrever` incrementa `trocas` em 1 por atribuição de um elemento a uma posição. Comparações de índices (por exemplo `j >= 0`, `i < n`) não são contadas. Para os algoritmos não comparativos, `comp` fica em zero (exceto no Bucket Sort, que usa Insertion Sort nos baldes). A tabela resume o que foi contado:

| Algoritmo | Comparações (`comp`) | Movimentações (`trocas`) |
|---|---|---|
| Bubble | `v[j+1] < v[j]` | cada *swap* |
| Selection | `v[j] < v[m]` na busca do mínimo | cada *swap* (só se $m \neq i$) |
| Insertion | `chave < v[j-1]`, apenas quando `j > inicio` | cada deslocamento e a escrita da chave na posição final, que só ocorre se a chave saiu do lugar |
| Shellsort | `chave < v[j-h]`, apenas quando `j >= h` | cada deslocamento e a escrita da chave na posição final, que só ocorre se a chave saiu do lugar |
| Merge | `aux[j] < aux[i]` na intercalação | cada escrita em `v[k]`; a cópia para `aux` **não** é contada |
| Quicksort | `v[j] <= pivô` e, em `mediana3`, as comparações de escolha do pivô | cada *swap* (troca do pivô para o fim, da partição e do pivô final); trocas de um elemento com ele mesmo não são feitas nem contadas |
| Counting | nenhuma | cada escrita no vetor de saída |
| Radix | nenhuma | cada escrita no vetor de saída, em cada passada |
| Bucket | as do Insertion Sort nos baldes | cada inserção em um balde, as movimentações do Insertion Sort nos baldes e cada escrita na saída |

**Algoritmos quadráticos.** O Bubble Sort usa a parada antecipada. O Selection Sort só troca quando o índice do mínimo difere de $i$. O Insertion Sort foi implementado por intervalo (`insertion_sort_intervalo(v, inicio, fim, c)`, com `fim` inclusivo), para ser reaproveitado no Bucket Sort.

**Shellsort.** A função recebe a sequência de *gaps* como parâmetro. Foram implementadas: **Shell** ($\lfloor n/2\rfloor, \lfloor n/4\rfloor, \dots, 1$); **Knuth** ($h_{k+1} = 3h_k + 1$, enquanto $h < n/3$, em ordem decrescente); e **Ciura**, com os termos empíricos $1, 4, 10, 23, 57, 132, 301, 701, 1750$ estendidos por $\lfloor 2{,}25\,h\rfloor$ enquanto o termo for menor que $n$. A versão principal é a de Ciura.

**Merge Sort.** Versão *top-down*, com um único vetor auxiliar de tamanho $n$ alocado uma vez e passado às chamadas recursivas. Os índices `lo` e `hi` são inclusivos: o meio é $\lfloor(lo + hi)/2\rfloor$ e as metades são `v[lo..mid]` e `v[mid+1..hi]`. Em caso de empate na intercalação, o elemento da metade esquerda é copiado primeiro (comparação estrita `aux[j] < aux[i]`), o que mantém a estabilidade.

**Quicksort.** Partição de **Lomuto** com o pivô na última posição e índices `lo` e `hi` inclusivos; elementos $\le$ pivô vão para a esquerda. Três estratégias de pivô, aplicadas trocando o pivô escolhido com `v[hi]` antes da partição: `"ultimo"` (índice `hi`), `"aleatorio"` (índice sorteado uniformemente em `[lo, hi]` com um `random.Random` próprio, criado com semente fixa 0 a cada chamada do algoritmo, o que torna a sequência de pivôs reproduzível para uma mesma entrada) e `"mediana3"` (mediana entre `v[lo]`, `v[(lo + hi) // 2]` e `v[hi]`, com 2 ou 3 comparações contadas). Trocas de um elemento com ele mesmo (pivô já em `v[hi]`, `i = j` na partição ou pivô final já na posição) não são feitas nem contadas. Para controlar a profundidade de recursão, o algoritmo chama a recursão apenas na **menor** partição e trata a maior com um laço, o que garante profundidade $O(\log n)$ mesmo no pior caso. Não foi implementada a partição de três vias; por isso, vetores com muitos valores repetidos degradam o Quicksort com Lomuto, e esse efeito é analisado no experimento E5. A versão principal é a de pivô aleatório.

**Counting Sort.** Usa $k = \max(v) + 1$, percorre a entrada de trás para frente para ser estável e devolve uma nova lista. Foi implementada também a variante `counting_sort_por_chave(v, k, chave, c)`, que ordena de forma estável por `chave(x)`, usada pelo Radix Sort.

**Radix Sort.** LSD, com uma passada do Counting Sort por dígito. A base padrão é $b = 2^8 = 256$; a base é parâmetro, e no experimento E6 comparam-se as bases $10$, $256$ e $65\,536$. Antes das passadas, os valores são deslocados pelo mínimo $m$ (dígito $\lfloor (x - m)/b^i\rfloor \bmod b$), o que permite valores negativos; o número de passadas é $d = \lfloor\log_b (M - m)\rfloor + 1$, em que $M$ é o maior valor. Nas entradas dos experimentos, $m$ é próximo de 0, e $d$ coincide com $\lfloor\log_b M\rfloor + 1$.

**Bucket Sort.** Cria $n$ baldes; o elemento $x \in [0,1)$ vai para o balde $\min(\lfloor n\cdot x\rfloor, n-1)$, limite que protege contra arredondamento; cada balde é ordenado com `insertion_sort_intervalo`; os baldes são concatenados em ordem, escrevendo de volta em `v`.

### 3.3 Verificação de corretude

Antes de qualquer experimento, todos os algoritmos passam por uma bateria de testes com casos-limite (vetor vazio, de um elemento, de dois elementos, com repetidos, já ordenado, inverso, aleatório, com valores grandes e com muitos valores repetidos), comparando a saída com o gabarito `sorted(entrada)`. Além disso, foram acrescentados três testes próprios. (i) **Estabilidade:** os algoritmos estáveis (Bubble, Insertion, Merge, Counting, Radix nas bases 256 e 10, e Bucket) ordenam 300 elementos com muitas chaves repetidas, cada um rotulado com sua posição original, e verifica-se que elementos de mesma chave mantêm a ordem dos rótulos. (ii) **Vetor já ordenado:** todos os algoritmos devem devolver o vetor inalterado, com e sem valores repetidos; para $n = 200$ sem repetidos conferem-se ainda as contagens exatas: $n-1$ comparações e nenhuma movimentação no Bubble e no Insertion Sort, $n(n-1)/2$ comparações e nenhuma troca no Selection Sort, nenhuma movimentação no Shellsort (nas três sequências) e $n(n-1)/2$ comparações no Quicksort com pivô no último elemento, que é o seu pior caso. (iii) **Casos de borda com vetores pequenos:** 26 vetores de até quatro elementos (vazio, um elemento, dois elementos nas duas ordens, todos iguais, valores distantes como $[1000, 0, 999]$ e todas as permutações de $[0, 1, 2]$ e de $[0, 1, 1, 2]$) são ordenados por todos os algoritmos e comparados com `sorted()`; para o Bucket Sort os valores são divididos por 1001, para ficarem em $[0, 1)$. Durante os experimentos, a saída de cada execução também é conferida com `esta_ordenado`, fora da janela de medição de tempo.

### 3.4 Geração das entradas

Para inteiros, as entradas são geradas por `gerar_inteiros(tipo, n, seed, k)` com valores em $[0, k)$ (por padrão $k = n$), em quatro tipos: **aleatória**; **ordenada**; **inversa**; e **quase ordenada** (vetor ordenado com cerca de 1% de pares trocados aleatoriamente). Os tipos ordenada e inversa são construídos diretamente, sem usar `sorted()`. Para o Bucket Sort, `gerar_floats(n, seed, distribuicao)` produz reais em $[0,1)$ com distribuição **uniforme** ou **concentrada** (90% dos valores em $[0, 0{,}1)$).

**Reprodutibilidade.** Cada geração usa um gerador próprio `random.Random(seed)`, de modo que o mesmo par (tipo, $n$, *seed*) sempre produz o mesmo vetor. Usam-se 5 *seeds* (11, 22, 33, 44 e 55). Para cada algoritmo, o vetor de cada combinação (tipo, $n$, *seed*) é gerado a partir da mesma *seed* e o algoritmo recebe uma **cópia** dele, de modo que todos ordenam exatamente a mesma entrada.

### 3.5 Experimentos e faixas de $n$

As faixas de $n$ foram calibradas pela complexidade de cada grupo de algoritmos, para que cada experimento termine em tempo razoável e ainda cubra pelo menos uma ordem de grandeza (necessário para enxergar o crescimento em escala log-log).

| Exp. | Algoritmos | Tipos de entrada | Valores de $n$ | *Seeds* |
|---|---|---|---|---|
| **E1** | os 9 (versões principais) | aleatória, ordenada, inversa, quase ordenada (Bucket: uniforme) | 100, 200, 500, 1 000, 2 000, 5 000 | 5 |
| **E2** | Bubble, Insertion, Selection | os 4 tipos | 500, 1 000, 2 000, 5 000, 10 000 | 5 |
| **E3** | Shell (3 *gaps*), Merge, Quick (3 pivôs) | os 4 tipos | $10^3$, $3\cdot10^3$, $10^4$, $3\cdot10^4$, $10^5$ | 5 |
| **E4** | Counting, Radix, Bucket; Merge e Quick (aleatório) como referência | aleatória com $k \in \{100, n, n^{1{,}5}\}$ (Bucket: uniforme e concentrada; referências: $k = n$) | $10^3$, $10^4$, $10^5$, $3\cdot10^5$, $10^6$ | 5 |
| **E5** | Quick (último, aleatório, mediana de 3) | ordenada, inversa, aleatória com $k = 10$ | 250, 500, 1 000, 2 000, 4 000 | 5 |
| **E6** | Radix (bases 10, 256, 65 536) | aleatória com $k = 10^9$ | $10^4$, $10^5$, $10^6$ | 5 |

Os experimentos foram executados no modo completo, com as faixas da tabela acima. Os tamanhos opcionais do E3 ($n = 3\cdot10^5$ e $10^6$) não foram incluídos.

### 3.6 Medição de tempo

O tempo é medido com `time.perf_counter()` (relógio de alta resolução), envolvendo **apenas** a chamada do algoritmo: a geração do vetor, a cópia e a verificação ficam de fora. O coletor de lixo (*garbage collector*) é executado antes (`gc.collect()`) e desativado durante a medição (`gc.disable()`), sendo reativado em seguida, para reduzir o ruído. Durante o modo completo, a máquina foi mantida sem outros programas pesados em execução. Para cada (algoritmo, tipo, $n$) registra-se uma linha por *seed* com o tempo, as comparações e as movimentações, e os resultados são salvos em arquivos CSV (`resultados_E1.csv` a `resultados_E6.csv`). A função `resumir` calcula a média e o desvio padrão entre as *seeds*.

**Mecanismo de proteção.** Cada execução tem um `tempo_limite` (20 s por padrão, 60 s no E2 e 15 s nas referências Merge e Quick do E4). Se uma execução (ou a média das *seeds*) ultrapassa o limite, ou se a previsão para o próximo tamanho, supondo crescimento quadrático a partir do tamanho anterior, ultrapassa 4 vezes o limite, os tamanhos maiores são pulados para aquele algoritmo e tipo. Na execução, três combinações foram interrompidas. (i) e (ii) `quick_ultimo` em entrada **ordenada** e **inversa** no E3, após $n = 10^4$: as médias foram de 10,5 s e 9,2 s, e a previsão quadrática para $n = 3\cdot10^4$ (cerca de 94 s e 83 s) ultrapassou 4 vezes o limite de 20 s, o que é coerente com o custo $\Theta(n^2)$ desse pior caso. (iii) Counting Sort com $k = n^{1{,}5}$ no E4, após $n = 3\cdot10^5$ ($k \approx 1{,}6\cdot10^8$): a primeira execução levou 22,2 s e ultrapassou o limite, de modo que esse tamanho tem uma única *seed* e $n = 10^6$ ($k = 10^9$) não foi executado. Nenhuma outra combinação foi interrompida.

### 3.7 Análise dos resultados

Os resultados são analisados com tabelas e gráficos de tempo, comparações e movimentações em função de $n$, em escala linear e log-log, com barras de erro de um desvio padrão e curvas teóricas ($n$, $n\log n$, $n^{1{,}5}$, $n^2$) escaladas para coincidir com o último ponto medido de um algoritmo de referência. Dois instrumentos quantitativos complementam a inspeção visual:

- **Expoente estimado $\alpha$:** ajusta-se por mínimos quadrados uma reta a $\log T$ versus $\log n$, usando as médias com $n \ge 1000$ de cada (algoritmo, tipo, $k$), reunindo os pontos de todos os experimentos, desde que haja pelo menos três tamanhos; a inclinação estima o expoente em $T(n)\approx c\,n^{\alpha}$. Para $n\log n$ espera-se $\alpha$ ligeiramente acima de 1 (da ordem de $1 + 1/\ln n$ numa faixa limitada).
- **Razão $T(n)/g(n)$:** se $T(n) = \Theta(g(n))$, a razão deve se estabilizar à medida que $n$ cresce, o que é um teste mais convincente do que a semelhança visual entre curvas.

### 3.8 Ameaças à validade

- **Instrumentação.** A chamada de funções auxiliares a cada comparação ou movimentação acrescenta um custo aproximadamente constante por operação. Isso altera as constantes, mas não a ordem de crescimento, e todos os algoritmos pagam esse custo; contudo, ele penaliza mais os algoritmos comparativos (que contam muito) do que os não comparativos, o que exagera a vantagem dos lineares.
- **Overhead do interpretador.** O Python é interpretado, e cada operação elementar custa dezenas de nanossegundos; os resultados absolutos não se transferem para linguagens compiladas.
- **Ruído.** Tempos abaixo de aproximadamente 1 ms são muito ruidosos. O ruído do sistema operacional, o cache e o gerenciamento de memória afetam as medições, e o uso de 5 *seeds* dá apenas uma estimativa grosseira da variabilidade. As contagens de comparações e movimentações, por serem determinísticas para uma dada entrada, são muito mais estáveis que o tempo.
- **Faixa de $n$ limitada.** As faixas foram calibradas para o tempo de execução em Python; para $n$ pequeno, termos de ordem inferior e constantes podem mascarar o comportamento assintótico, e os expoentes estimados são aparentes.
- **Decisões de implementação.** A partição de Lomuto degrada com valores repetidos, e a escolha de contar ou não as cópias para o vetor auxiliar do Merge Sort altera as movimentações reportadas; ambas as decisões estão documentadas na Seção 3.2.
- **Uma única máquina.** Os resultados refletem um único ambiente de hardware e software.

### 3.9 Como reproduzir

Com Python 3.10 ou superior e as bibliotecas `numpy`, `pandas` e `matplotlib`, basta abrir o notebook `TP_Algoritmos_de_Ordenacao.ipynb`, executar **Kernel → Restart & Run All** e aguardar o término dos experimentos. Pela linha de comando, o mesmo resultado é obtido com `papermill TP_Algoritmos_de_Ordenacao.ipynb TP_executado.ipynb --log-output`. Como toda aleatoriedade vem de `random.Random(seed)`, com as *seeds* fixas acima na geração das entradas e a semente 0 no pivô aleatório do Quicksort, as contagens de comparações e movimentações são reproduzidas exatamente; os tempos variam ligeiramente conforme a máquina.

## 4. Resultados
*escreva aqui (referencie os gráficos G1–G8 e as tabelas T1–T2)*

## 5. Análise crítica
*escreva aqui*

## 6. Conclusões
*escreva aqui*

## Referências

CIURA, M. Best increments for the average case of Shellsort. In: FREIVALDS, R. (ed.). **Fundamentals of Computation Theory (FCT 2001)**. Berlin: Springer, 2001. p. 106–117. (Lecture Notes in Computer Science, v. 2138).

CORMEN, T. H.; LEISERSON, C. E.; RIVEST, R. L.; STEIN, C. **Algoritmos: teoria e prática**. 3. ed. Rio de Janeiro: Elsevier, 2012.

HOARE, C. A. R. Quicksort. **The Computer Journal**, Oxford, v. 5, n. 1, p. 10–16, 1962.

KNUTH, D. E. **The art of computer programming, volume 3: sorting and searching**. 2. ed. Reading: Addison-Wesley, 1998.

PRATT, V. R. **Shellsort and sorting networks**. New York: Garland, 1979.

SEDGEWICK, R.; WAYNE, K. **Algorithms**. 4. ed. Upper Saddle River: Addison-Wesley, 2011.

SHELL, D. L. A high-speed sorting procedure. **Communications of the ACM**, New York, v. 2, n. 7, p. 30–32, 1959.

ZIVIANI, N. **Projeto de algoritmos: com implementações em Java e C++**. São Paulo: Thomson Learning, 2007.